In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import json
import random
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/"
    "football-adboard-segmentation"
)

TEST_DIR = (
    PROJECT_DIR
    / "datasets"
    / "rf_detr"
    / "test"
)

TEST_JSON = TEST_DIR / "_annotations.coco.json"

print("Test:", TEST_DIR)
print("JSON:", TEST_JSON)

print("\nControlli:")
print("Test esiste:", TEST_DIR.exists())
print("JSON esiste:", TEST_JSON.exists())

Test: /content/drive/MyDrive/football-adboard-segmentation/datasets/rf_detr/test
JSON: /content/drive/MyDrive/football-adboard-segmentation/datasets/rf_detr/test/_annotations.coco.json

Controlli:
Test esiste: True
JSON esiste: True


In [3]:
with TEST_JSON.open("r", encoding="utf-8") as f:
    test_coco = json.load(f)

print("Immagini totali test:", len(test_coco["images"]))
print("Annotazioni totali test:", len(test_coco["annotations"]))

random.seed(42)

REVIEW_SIZE = 100

review_images = random.sample(
    test_coco["images"],
    REVIEW_SIZE
)

print("\nImmagini selezionate:", len(review_images))

for info in review_images[:10]:
    print("-", info["file_name"])

Immagini totali test: 1327
Annotazioni totali test: 2939

Immagini selezionate: 100
- pn7hk7kglff97qh.png
- 9fsjhvuxzmqknes.png
- 88a0iezjzlukcuk.png
- 0xkr4fm2tlukxe4.png
- qmhdyyfe7r763jr.png
- 8smpjqidz6sxq85.png
- 2hsur0xgzyg1nhr.png
- si1cuhadbbi479r.png
- rl3ndjvrej0c8yz.png
- 3hcope86l3s3vk9.png


In [4]:
REVIEW_DIR = (
    PROJECT_DIR
    / "outputs"
    / "reviewed_test_subset_100"
)

REVIEW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

manifest = pd.DataFrame([
    {
        "image_id": info["id"],
        "file_name": info["file_name"]
    }
    for info in review_images
])

MANIFEST_PATH = REVIEW_DIR / "subset_manifest.csv"

manifest.to_csv(
    MANIFEST_PATH,
    index=False
)

print("Manifest salvato:")
print(MANIFEST_PATH)

display(manifest.head(10))

Manifest salvato:
/content/drive/MyDrive/football-adboard-segmentation/outputs/reviewed_test_subset_100/subset_manifest.csv


,image_id,file_name
0,1310,pn7hk7kglff97qh.png
1,229,9fsjhvuxzmqknes.png
2,52,88a0iezjzlukcuk.png
3,564,0xkr4fm2tlukxe4.png
4,502,qmhdyyfe7r763jr.png
5,458,8smpjqidz6sxq85.png
6,286,2hsur0xgzyg1nhr.png
7,210,si1cuhadbbi479r.png
8,1117,rl3ndjvrej0c8yz.png
9,179,3hcope86l3s3vk9.png


In [5]:
import json
import shutil
from pathlib import Path

# Cartella del mini-dataset che revisioneremo
REVIEW_DATASET_DIR = REVIEW_DIR / "original_gt"

if REVIEW_DATASET_DIR.exists():
    shutil.rmtree(REVIEW_DATASET_DIR)

REVIEW_DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ID delle 100 immagini selezionate
selected_ids = {
    info["id"]
    for info in review_images
}

# Filtriamo immagini e annotazioni COCO
subset_images = [
    img
    for img in test_coco["images"]
    if img["id"] in selected_ids
]

subset_annotations = [
    ann
    for ann in test_coco["annotations"]
    if ann["image_id"] in selected_ids
]

subset_coco = {
    "images": subset_images,
    "annotations": subset_annotations,
    "categories": test_coco["categories"]
}

# Copia delle 100 immagini
for img_info in subset_images:
    src = TEST_DIR / img_info["file_name"]
    dst = REVIEW_DATASET_DIR / img_info["file_name"]

    assert src.exists(), f"Immagine mancante: {src}"

    shutil.copy2(src, dst)

# Salvataggio COCO originale del subset
SUBSET_JSON = (
    REVIEW_DATASET_DIR
    / "_annotations.coco.json"
)

with SUBSET_JSON.open("w", encoding="utf-8") as f:
    json.dump(
        subset_coco,
        f,
        indent=2
    )

print("Subset creato:")
print(REVIEW_DATASET_DIR)

print("\nImmagini:", len(subset_images))
print("Annotazioni originali:", len(subset_annotations))
print("Categorie:", subset_coco["categories"])

Subset creato:
/content/drive/MyDrive/football-adboard-segmentation/outputs/reviewed_test_subset_100/original_gt

Immagini: 100
Annotazioni originali: 217
Categorie: [{'id': 1, 'name': 'advertising_board', 'supercategory': 'advertising_board'}]


In [6]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}

physical_images = [
    p
    for p in REVIEW_DATASET_DIR.iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
]

print("Immagini fisiche:", len(physical_images))
print("Immagini nel COCO:", len(subset_coco["images"]))
print("Annotazioni:", len(subset_coco["annotations"]))

assert len(physical_images) == 100
assert len(subset_coco["images"]) == 100

print("\n✓ Subset da revisionare pronto")

Immagini fisiche: 100
Immagini nel COCO: 100
Annotazioni: 217

✓ Subset da revisionare pronto


In [7]:
import shutil

ZIP_BASE = REVIEW_DIR / "review_test_100_roboflow"

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=REVIEW_DATASET_DIR
)

print("ZIP creato:")
print(zip_path)

ZIP creato:
/content/drive/MyDrive/football-adboard-segmentation/outputs/reviewed_test_subset_100/review_test_100_roboflow.zip


In [8]:
from pathlib import Path

zip_file = Path(zip_path)

print("Esiste:", zip_file.exists())
print(f"Dimensione: {zip_file.stat().st_size / (1024**2):.1f} MB")

Esiste: True
Dimensione: 80.6 MB


In [9]:
from pathlib import Path

REVISED_ROOT = Path(
    "/content/drive/MyDrive/"
    "football-adboard-segmentation/datasets/reviewed_test_100"
)

print("Contenuto:")
for p in REVISED_ROOT.rglob("*"):
    if p.name.endswith(".json"):
        print(p)

Contenuto:
/content/drive/MyDrive/football-adboard-segmentation/datasets/reviewed_test_100/_annotations.coco.json


In [10]:
REVISED_JSON = (
    REVISED_ROOT
    / "test"
    / "_annotations.coco.json"
)

In [12]:
from pathlib import Path

REVISED_ROOT = Path(
    "/content/drive/MyDrive/"
    "football-adboard-segmentation/datasets/reviewed_test_100"
)

json_files = list(REVISED_ROOT.rglob("_annotations.coco.json"))

print("JSON trovati:", len(json_files))

for p in json_files:
    print(p)

JSON trovati: 1
/content/drive/MyDrive/football-adboard-segmentation/datasets/reviewed_test_100/_annotations.coco.json


In [13]:
import json

REVISED_JSON = json_files[0]

with REVISED_JSON.open("r", encoding="utf-8") as f:
    revised_coco = json.load(f)

print("Immagini:", len(revised_coco["images"]))
print("Annotazioni revisionate:", len(revised_coco["annotations"]))
print("Categorie:", revised_coco["categories"])

Immagini: 100
Annotazioni revisionate: 338
Categorie: [{'id': 0, 'name': 'Football-Adboards-Reviewed-Tes', 'supercategory': 'none'}, {'id': 1, 'name': 'advertising_board', 'supercategory': 'Football-Adboards-Reviewed-Tes'}]


In [14]:
ORIGINAL_ANNOTATIONS = 217
REVISED_ANNOTATIONS = len(revised_coco["annotations"])

added = REVISED_ANNOTATIONS - ORIGINAL_ANNOTATIONS

print("Annotazioni originali :", ORIGINAL_ANNOTATIONS)
print("Annotazioni revisionate:", REVISED_ANNOTATIONS)
print("Nuove box aggiunte      :", added)
print(
    f"Aumento annotazioni     : "
    f"{100 * added / ORIGINAL_ANNOTATIONS:.1f}%"
)

Annotazioni originali : 217
Annotazioni revisionate: 338
Nuove box aggiunte      : 121
Aumento annotazioni     : 55.8%


In [15]:
from pathlib import Path
import shutil

REVISED_ROOT = Path(
    "/content/drive/MyDrive/"
    "football-adboard-segmentation/datasets/reviewed_test_100"
)

LOCAL_REVISED = Path("/content/reviewed_test_100_eval")
LOCAL_TEST = LOCAL_REVISED / "test"

if LOCAL_REVISED.exists():
    shutil.rmtree(LOCAL_REVISED)

LOCAL_TEST.mkdir(parents=True)

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}

# Copia immagini
for path in REVISED_ROOT.iterdir():
    if path.suffix.lower() in IMAGE_EXTENSIONS:
        shutil.copy2(path, LOCAL_TEST / path.name)

# Copia COCO
shutil.copy2(
    REVISED_ROOT / "_annotations.coco.json",
    LOCAL_TEST / "_annotations.coco.json"
)

print("Dataset preparato:", LOCAL_REVISED)

Dataset preparato: /content/reviewed_test_100_eval


In [17]:
from collections import Counter

category_counts = Counter(
    ann["category_id"]
    for ann in check_coco["annotations"]
)

print("Annotazioni per category_id:")
for category_id, count in sorted(category_counts.items()):
    print(f"category_id {category_id}: {count}")

print("\nCategorie dichiarate:")
for category in check_coco["categories"]:
    print(category)

Annotazioni per category_id:
category_id 1: 338

Categorie dichiarate:
{'id': 0, 'name': 'Football-Adboards-Reviewed-Tes', 'supercategory': 'none'}
{'id': 1, 'name': 'advertising_board', 'supercategory': 'Football-Adboards-Reviewed-Tes'}


In [19]:
import json

LOCAL_JSON = LOCAL_TEST / "_annotations.coco.json"

with LOCAL_JSON.open("r", encoding="utf-8") as f:
    coco_fixed = json.load(f)

# Manteniamo solo la classe realmente usata
coco_fixed["categories"] = [
    {
        "id": 1,
        "name": "advertising_board",
        "supercategory": "none"
    }
]

with LOCAL_JSON.open("w", encoding="utf-8") as f:
    json.dump(coco_fixed, f, indent=2)

print("✓ JSON locale corretto")

✓ JSON locale corretto


In [20]:
with LOCAL_JSON.open("r", encoding="utf-8") as f:
    check_fixed = json.load(f)

print("Immagini:", len(check_fixed["images"]))
print("Annotazioni:", len(check_fixed["annotations"]))
print("Categorie:", check_fixed["categories"])

Immagini: 100
Annotazioni: 338
Categorie: [{'id': 1, 'name': 'advertising_board', 'supercategory': 'none'}]


In [21]:
import json

with (LOCAL_TEST / "_annotations.coco.json").open(
    "r",
    encoding="utf-8"
) as f:
    check_coco = json.load(f)

physical_images = [
    p for p in LOCAL_TEST.iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
]

print("Immagini fisiche :", len(physical_images))
print("Immagini COCO    :", len(check_coco["images"]))
print("Annotazioni COCO :", len(check_coco["annotations"]))
print("Categorie        :", check_coco["categories"])

Immagini fisiche : 100
Immagini COCO    : 100
Annotazioni COCO : 338
Categorie        : [{'id': 1, 'name': 'advertising_board', 'supercategory': 'none'}]


In [22]:
!pip install -q "rfdetr[train]" supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.3/373.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.9/587.9 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.5/530.5 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.5 MB/s eta 0:00:00


In [23]:
import torch

print("CUDA disponibile:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA disponibile: False


In [24]:
from pathlib import Path
from rfdetr import RFDETRNano

CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/"
    "football-adboard-segmentation/"
    "outputs/rfdetr_nano_smoke_test_local/"
    "checkpoint_best_total.pth"
)

print("Checkpoint:", CHECKPOINT_PATH)
print("Esiste:", CHECKPOINT_PATH.exists())

model = RFDETRNano.from_checkpoint(
    str(CHECKPOINT_PATH)
)

print("✓ Modello finale caricato")

Checkpoint: /content/drive/MyDrive/football-adboard-segmentation/outputs/rfdetr_nano_smoke_test_local/checkpoint_best_total.pth
Esiste: True


/usr/local/lib/python3.13/dist-packages/rfdetr/detr.py:551: FutureWarning: The `RFDETRBase` was deprecated since v1.7.0. It will be removed in v2.0.0.
  getattr(variant_obj, "__name__", symbol): variant_obj
/usr/local/lib/python3.13/dist-packages/rfdetr/detr.py:551: FutureWarning: The `RFDETRLargeDeprecated` was deprecated since v1.7.0. It will be removed in v2.0.0.
  getattr(variant_obj, "__name__", symbol): variant_obj
/usr/local/lib/python3.13/dist-packages/rfdetr/detr.py:551: FutureWarning: The `RFDETRSegPreview` was deprecated since v1.7.0. It will be removed in v2.0.0.
  getattr(variant_obj, "__name__", symbol): variant_obj
[2026-08-21 17:39:41] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-21 17:39:41] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is

✓ Modello finale caricato


In [26]:
import shutil
from pathlib import Path

LOCAL_REVISED = Path("/content/reviewed_test_100_eval")
LOCAL_TEST = LOCAL_REVISED / "test"
LOCAL_TRAIN = LOCAL_REVISED / "train"

if LOCAL_TRAIN.exists():
    shutil.rmtree(LOCAL_TRAIN)

shutil.copytree(
    LOCAL_TEST,
    LOCAL_TRAIN
)

print("Struttura preparata:")
print("Train JSON:", (LOCAL_TRAIN / "_annotations.coco.json").exists())
print("Test JSON :", (LOCAL_TEST / "_annotations.coco.json").exists())

Struttura preparata:
Train JSON: True
Test JSON : True


In [27]:
import json

with (LOCAL_TEST / "_annotations.coco.json").open(
    "r",
    encoding="utf-8"
) as f:
    revised_check = json.load(f)

print("Test images      :", len(revised_check["images"]))
print("Test annotations :", len(revised_check["annotations"]))
print("Categories       :", revised_check["categories"])

Test images      : 100
Test annotations : 338
Categories       : [{'id': 1, 'name': 'advertising_board', 'supercategory': 'none'}]


In [28]:
REVIEWED_DATASET = Path(
    "/content/reviewed_test_100_eval"
)

reviewed_metrics = model.evaluate(
    dataset_dir=str(REVIEWED_DATASET),
    split="test",
    batch_size=4,
)

print("\n================================")
print(" REVIEWED TEST SET - 100 IMMAGINI")
print("================================\n")

for key, value in reviewed_metrics.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

[2026-08-21 17:41:22] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-21 17:41:22] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-08-21 17:41:23] [INFO] rf-detr - Building Roboflow test dataset with square resize at resolution 384
[2026-08-21 17:41:23] [INFO] rf-detr - Using multi-scale training with square resize and scales: [544]


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ test/AP/advertising_board │    0.5804449319839478     │
│          test/F1          │    0.7827647924423218     │
│         test/loss         │     3.518444776535034     │
│        test/mAP_50        │    0.7098044157028198     │
│      test/mAP_50_95       │    0.5804449319839478     │
│        test/mAP_75        │    0.6419232487678528     │
│         test/mAR          │    0.6781064867973328     │
│      test/precision       │    0.9954338073730469     │
│        test/recall        │    0.6449704170227051     │
└───────────────────────────┴───────────────────────────┘


 REVIEWED TEST SET - 100 IMMAGINI

test/loss: 3.5184
test/mAP_50_95: 0.5804
test/mAP_50: 0.7098
test/mAP_75: 0.6419
test/mAR: 0.6781
test/F1: 0.7828
test/precision: 0.9954
test/recall: 0.6450
test/AP/advertising_board: 0.5804


In [29]:
from pathlib import Path
import shutil

ORIGINAL_GT = Path(
    "/content/drive/MyDrive/"
    "football-adboard-segmentation/"
    "outputs/reviewed_test_subset_100/"
    "original_gt"
)

LOCAL_ORIGINAL = Path("/content/original_test_100_eval")
LOCAL_ORIGINAL_TEST = LOCAL_ORIGINAL / "test"
LOCAL_ORIGINAL_TRAIN = LOCAL_ORIGINAL / "train"

print("Original GT esiste:", ORIGINAL_GT.exists())

Original GT esiste: True


In [30]:
if LOCAL_ORIGINAL.exists():
    shutil.rmtree(LOCAL_ORIGINAL)

# test vero
shutil.copytree(
    ORIGINAL_GT,
    LOCAL_ORIGINAL_TEST
)

# copia tecnica per far riconoscere il formato COCO a RF-DETR
shutil.copytree(
    ORIGINAL_GT,
    LOCAL_ORIGINAL_TRAIN
)

print("Train JSON:",
      (LOCAL_ORIGINAL_TRAIN / "_annotations.coco.json").exists())

print("Test JSON:",
      (LOCAL_ORIGINAL_TEST / "_annotations.coco.json").exists())

Train JSON: True
Test JSON: True


In [31]:
import json

with (
    LOCAL_ORIGINAL_TEST / "_annotations.coco.json"
).open("r", encoding="utf-8") as f:
    original_100_coco = json.load(f)

print("Immagini:", len(original_100_coco["images"]))
print("Annotazioni:", len(original_100_coco["annotations"]))
print("Categorie:", original_100_coco["categories"])

Immagini: 100
Annotazioni: 217
Categorie: [{'id': 1, 'name': 'advertising_board', 'supercategory': 'advertising_board'}]


In [32]:
original_100_metrics = model.evaluate(
    dataset_dir=str(LOCAL_ORIGINAL),
    split="test",
    batch_size=4,
)

print("\n================================")
print(" ORIGINAL GT - STESSE 100 IMMAGINI")
print("================================\n")

for key, value in original_100_metrics.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

[2026-08-21 17:47:01] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-21 17:47:01] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-08-21 17:47:02] [INFO] rf-detr - Building Roboflow test dataset with square resize at resolution 384
[2026-08-21 17:47:02] [INFO] rf-detr - Using multi-scale training with square resize and scales: [544]


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ test/AP/advertising_board │    0.8880401849746704     │
│          test/F1          │    0.9977011680603027     │
│         test/loss         │    1.5068747997283936     │
│        test/mAP_50        │    0.9999545812606812     │
│      test/mAP_50_95       │    0.8880401849746704     │
│        test/mAP_75        │    0.9816373586654663     │
│         test/mAR          │     0.929953932762146     │
│      test/precision       │    0.9954128265380859     │
│        test/recall        │            1.0            │
└───────────────────────────┴───────────────────────────┘


 ORIGINAL GT - STESSE 100 IMMAGINI

test/loss: 1.5069
test/mAP_50_95: 0.8880
test/mAP_50: 1.0000
test/mAP_75: 0.9816
test/mAR: 0.9300
test/F1: 0.9977
test/precision: 0.9954
test/recall: 1.0000
test/AP/advertising_board: 0.8880


In [33]:
import json

COMPARISON_PATH = (
    PROJECT_DIR
    / "outputs"
    / "reviewed_test_subset_100"
    / "rfdetr_gt_comparison.json"
)

comparison = {
    "num_images": 100,
    "original_annotations": 217,
    "revised_annotations": 338,
    "added_annotations": 121,
    "original_gt": {
        "mAP_50_95": 0.8880401849746704,
        "mAP_50": 0.9999545812606812,
        "mAP_75": 0.9816373586654663,
        "mAR": 0.929953932762146,
        "F1": 0.9977011680603027,
        "precision": 0.9954128265380859,
        "recall": 1.0
    },
    "revised_gt": {
        "mAP_50_95": 0.5804449319839478,
        "mAP_50": 0.7098044157028198,
        "mAP_75": 0.6419232487678528,
        "mAR": 0.6781064867973328,
        "F1": 0.7827647924423218,
        "precision": 0.9954338073730469,
        "recall": 0.6449704170227051
    }
}

COMPARISON_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

with COMPARISON_PATH.open("w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=4)

print("Confronto salvato in:")
print(COMPARISON_PATH)

Confronto salvato in:
/content/drive/MyDrive/football-adboard-segmentation/outputs/reviewed_test_subset_100/rfdetr_gt_comparison.json
